In [7]:
# BASE모델 , 파인튜닝 Instruct 성능 비교
# Qween2.5  Qween2.5_instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = '프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘'
base_model_id = 'Qwen/Qwen2.5-0.5B'
instruct_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

def generate(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.1,
        do_sample=True,
        max_new_tokens=40
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
base_output = generate(base_model_id)
print(f'[Base 모델 출력 : {base_output}]')
instruct_output = generate(instruct_model_id)
print(f'[Instruct 모델 출력 : {instruct_output}]')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Base 모델 출력 : 프랑스 파리의 명소를 추천해드리겠습니다. 1. 파리의 유명한 명소 중 하나는 "파리의 유명한 명소 중 하나]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]